# TrOCR Kannada Real-Data Retraining (Google Colab T4 GPU)

This notebook trains TrOCR on the authentic **IIIT-INDIC-HW-WORDS Kannada** dataset (102,999 real handwritten word images) combined with **15x oversampled synthetic conjunct-rich handwriting lines** (~109,000 total samples).

### Key Reliability & Performance Features
1. **Zero Benchmark Contamination**: All 13 archival crops (`doc1.jpeg` lines + personal trial) are **100% excluded** from training and validation, and reserved strictly for honest post-training held-out evaluation.
2. **Preserved Multi-Word Spacing**: Synthetic conjunct lines are oversampled 15x in `train.jsonl` (6,000 synthetic lines vs 73,517 real words = **7.55% synthetic**), preserving space and ligature segmentation without diluting real handwriting features.
3. **Google Drive Checkpointing**: Intermediate checkpoints are saved directly to `/content/drive/MyDrive/trocr_kannada_checkpoints` every 1,000 steps. If Colab disconnects, you resume seamlessly rather than restart.
4. **Local NVMe Staging**: `kn.zip` is copied from Google Drive to local `/content/` and extracted locally. This avoids FUSE filesystem latency over 100k individual JPEG reads.
5. **Latin Token Suppression**: Eliminates English decoder priors so TrOCR cannot hallucinate Latin characters (`SpaceX`, `HMRC`, etc.).

### Recommended Runtime
Go to **Runtime > Change runtime type** and select **T4 GPU** (Free tier).

In [ ]:
# 1. Hardware & Environment Check
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (Required for persistent checkpoints & kn.zip)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Install Python Dependencies
!pip install -q transformers datasets evaluate jiwer torchvision pillow albumentations accelerate

In [ ]:
# 4. Install Kannada Fonts
!apt-get update -qq
!apt-get install -y -qq fonts-knda fonts-lohit-knda fonts-noto-cjk

In [ ]:
# 5. Clone or Setup Repository
%cd /content
# If cloning from GitHub:
# !git clone https://github.com/your-repo/land-record-digitization.git
# %cd land-record-digitization

# Verify current working directory
!pwd

In [ ]:
# 6. Copy kn.zip from Google Drive & Extract Locally (Fast NVMe)
import os
import shutil
from pathlib import Path

DRIVE_ZIP = Path('/content/drive/MyDrive/kn.zip')
LOCAL_ZIP = Path('/content/kn.zip')

assert DRIVE_ZIP.exists(), f"kn.zip not found at {DRIVE_ZIP}! Please upload kn.zip to your Google Drive root."

if not LOCAL_ZIP.exists():
    print(f"Copying {DRIVE_ZIP} to local SSD {LOCAL_ZIP} (~4.8GB, ~30-60s) ...")
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print("Copy complete!")
else:
    print("Local kn.zip already present.")

# Run automated extraction script into external_datasets/iiit_indic_hw_words/
!python training/scripts/extract_kn_dataset.py

In [ ]:
# 7. Merge Real Words + 15x Oversampled Synthetic Lines (Archival strictly excluded!)
!python training/scripts/prepare_real_handwriting_dataset.py \
    --output-dir training/datasets/combined_handwriting \
    --iiit-root external_datasets/iiit_indic_hw_words \
    --synthetic-dir training/datasets/synthetic_lines \
    --synthetic-oversample 15

In [ ]:
# 8. Launch TrOCR Retraining with Checkpoints to Google Drive
# Epochs: 2 (recommended, ~60-70 minutes total on T4)
# Checkpoints are saved to Google Drive every 1000 steps

!python training/train_trocr_kannada_gpu.py \
    --train-manifest training/datasets/combined_handwriting/train.jsonl \
    --val-manifest training/datasets/combined_handwriting/val.jsonl \
    --base-model microsoft/trocr-small-handwritten \
    --output-dir /content/drive/MyDrive/trocr_kannada_checkpoints \
    --epochs 2 \
    --batch-size 8 \
    --lr 3e-5 \
    --fp16 \
    --eval-steps 1000 \
    --save-steps 1000 \
    --save-total-limit 3 \
    --max-eval-samples 500

In [ ]:
# 9. RESUME CELL (Run ONLY if Colab disconnected during training)
import glob
checkpoints = sorted(glob.glob('/content/drive/MyDrive/trocr_kannada_checkpoints/checkpoint-*'))
if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"Resuming from: {latest_ckpt}")
    !python training/train_trocr_kannada_gpu.py \
        --train-manifest training/datasets/combined_handwriting/train.jsonl \
        --val-manifest training/datasets/combined_handwriting/val.jsonl \
        --base-model microsoft/trocr-small-handwritten \
        --output-dir /content/drive/MyDrive/trocr_kannada_checkpoints \
        --epochs 2 \
        --batch-size 8 \
        --lr 3e-5 \
        --fp16 \
        --eval-steps 1000 \
        --save-steps 1000 \
        --save-total-limit 3 \
        --max-eval-samples 500 \
        --resume-from-checkpoint "{latest_ckpt}"
else:
    print("No intermediate checkpoint found in Drive.")

In [ ]:
# 10. Honest Post-Training Evaluation on Contamination-Free Archival Benchmark
# Evaluates on all 13 archival crops (zero seen during training or validation)
!python training/scripts/run_real_data_trocr_experiment.py

In [ ]:
# 11. Package & Download the Best Checkpoint (also already safely saved on Drive)
!zip -r /content/kannada_trocr_best_checkpoint.zip /content/drive/MyDrive/trocr_kannada_checkpoints/best_checkpoint

from google.colab import files
files.download('/content/kannada_trocr_best_checkpoint.zip')